In [ ]:
# import os
# os.environ["OPENAI_API_KEY"] = ""

## Structured RAG

In [183]:
# 데이터가 제대로 읽혀지는지 확인
import pandas as pd
df = pd.read_csv('customer_db.csv')

print(df[df["name"] == '김민수']) # 조건 true 인것 찾기


   customer_id name  age marital_status  children_count  income region  \
0            1  김민수   35             기혼               2    6200     서울   

   has_house  has_disabled_member  subscription_years  savings_count  \
0      False                False                   8             96   

  account_type  
0     주택청약종합저축  


In [184]:
# Pandas Query에 대한 이해
# 기존 쿼리 처리방법 df[df['name'] == '김민수']

# 자녀수 2명 이상 + 무주택 + 소득 6천만원 이상인 고객
df_2 = df.query("children_count >= 2 and has_house == False and income > 6000")

print(df_2)


   customer_id name  age marital_status  children_count  income region  \
0            1  김민수   35             기혼               2    6200     서울   
2            3  박영희   40             기혼               3    7000     대전   
9           10  홍지현   41             기혼               3    6900     부산   

   has_house  has_disabled_member  subscription_years  savings_count  \
0      False                False                   8             96   
2      False                False                  10            120   
9      False                False                  11            132   

  account_type  
0     주택청약종합저축  
2     주택청약종합저축  
9     주택청약종합저축  


In [185]:
!pip install -q langchain langchain_community langchain_openai langchain-experimental chromadb tiktoken

In [186]:
from langchain_experimental.agents.agent_toolkits.pandas.base import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI



In [187]:
llm = ChatOpenAI(model = 'gpt-4o-mini')

In [188]:
# 에이전트의 기본 성격/규칙을 세팅하는 프롬프트
PREFIX = (
    "너는 pandas만 사용해 df(DataFrame)에서 답을 구해. "
    "한 줄 요약으로만 답하고, 표/그래프는 생성하지 마. "
    "실행이 실패하면 즉시 간단한 요약으로 fallback해."
)

In [189]:
# Pandas Agent 생성
agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose = True,
    allow_dangerous_code = True,
    max_iterations = 15,
    max_execution_time= 60,
    prefix = PREFIX
)

In [190]:
# 질의 함수
def ask(q):
  res = agent.invoke(q)
  return res if isinstance(res, str) else res.get("output", str(res))

In [191]:
print(ask("김민수의 나이, 결혼상태, 소득, 자녀수를 한줄로 요약해줘"))



> Entering new AgentExecutor chain...
Thought: 김민수의 정보를 df에서 찾아야 한다. 고객 ID로 필터링하면 필요한 정보를 추출할 수 있다.  
Action: python_repl_ast  
Action Input: df[df['name'] == '김민수'][['age', 'marital_status', 'income', 'children_count']]     age marital_status  income  children_count
0   35             기혼    6200               2I now know the final answer  
Final Answer: 김민수는 35세, 기혼이며, 소득은 6200이고 자녀는 2명이다.

> Finished chain.
김민수는 35세, 기혼이며, 소득은 6200이고 자녀는 2명이다.


In [192]:
PII_BLOCK = {"연락처", "주민", "이메일", "전화", "주소"}

def safe_ask(q):
  if any(k in q for k in PII_BLOCK): # PII안에 어떤것이 있더라도
    return "개인정보 항목은 조회할 수 없습니다."
  return ask(q)


In [193]:
print(safe_ask("김민수 고객의 주소를 알려줘"))

개인정보 항목은 조회할 수 없습니다.


##Structured RAG 실습

In [194]:
# 1. 직원 데이터 베이스 import
df = pd.read_csv("employee_db.csv")

In [195]:
# 2. 정책 문서 import (md, txt, docx, hwp, excel, csv, parquet 등)
from langchain_community.document_loaders import TextLoader

docs = TextLoader('company_policy.md', encoding = 'utf-8').load()
print(len(docs))
print(docs[0])

1
page_content='# 회사 내규 (요약본) — 휴가·연봉상승률·연차 규정

> 본 문서는 실습용 요약본입니다. 실제 정책과 다를 수 있습니다.

## 1. 직급 체계
- 사원 → 대리 → 과장 → 차장 → 부장

## 2. 연차 휴가 (기본)
- 사원: 연 15일
- 대리: 연 16일
- 과장: 연 18일
- 차장: 연 20일
- 부장: 연 22일

### 2.1 근속 가산
- 근속 3년 이상: +1일
- 근속 5년 이상: +2일
- 근속 10년 이상: +4일 (중복 불가, 최댓값 기준으로 적용)

### 2.2 이월 규정
- 미사용 연차 최대 5일 이월 가능 (차년도 6월 30일까지 사용)

### 2.3 특별휴가
- 출산(본인): 90일(법정 기준에 따름) — 별도 관리
- 배우자 출산: 10일
- 자녀 돌봄: 연 3일 (미성년 자녀 보유자만)

## 3. 연봉 상승률 (기본 가이드)
- 사원: 3~5%
- 대리: 4~6%
- 과장: 5~7%
- 차장: 4~6%
- 부장: 3~5%

### 3.1 성과 가중치 (Performance Multiplier)
- A: +2%p
- B: +0%p
- C: -1%p (단, 하한 0% 보장)

> 예) 대리(B)의 기본 4~6% → B는 +0%p이므로 4~6% 유지
> 예) 과장(A)의 기본 5~7% → A는 +2%p → 7~9%

## 4. 승진(프로모션) 참고
- 사원→대리: 최소 근속 2년 & 최근 2개년 평균 평가 B 이상
- 대리→과장: 최소 근속 4년 & 최근 2개년 평균 평가 B 이상
- 과장→차장: 최소 근속 5년 & 최근 2개년 평균 평가 B 이상
- 차장→부장: 최소 근속 6년 & 최근 3개년 평균 평가 B 이상

## 5. 계약직/원격
- 계약직: 연차 11일(비례 적용 가능), 성과가중치 최대 +0%p
- 완전 원격: 연차 동일, 출근 의무 없음(팀 합의 우선)

## 6. FAQ 키워드
- "연차", "이월", "자녀", "배우자 출산", "연봉 상승률", "성과", "승진", "계약

In [196]:
# 3. 정책문서에 대한 임베딩

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter


In [197]:
# chunk size = 700(500~1000) chunk_overlap (50~300)
splitter = RecursiveCharacterTextSplitter(chunk_size = 700, chunk_overlap = 100) # 문서가 짧아서 청크 사이즈 300 정도도 괜찮을듯?
splits = splitter.split_documents(docs)

In [198]:
print(splits)


[Document(metadata={'source': 'company_policy.md'}, page_content='# 회사 내규 (요약본) — 휴가·연봉상승률·연차 규정\n\n> 본 문서는 실습용 요약본입니다. 실제 정책과 다를 수 있습니다.\n\n## 1. 직급 체계\n- 사원 → 대리 → 과장 → 차장 → 부장\n\n## 2. 연차 휴가 (기본)\n- 사원: 연 15일\n- 대리: 연 16일\n- 과장: 연 18일\n- 차장: 연 20일\n- 부장: 연 22일\n\n### 2.1 근속 가산\n- 근속 3년 이상: +1일\n- 근속 5년 이상: +2일\n- 근속 10년 이상: +4일 (중복 불가, 최댓값 기준으로 적용)\n\n### 2.2 이월 규정\n- 미사용 연차 최대 5일 이월 가능 (차년도 6월 30일까지 사용)\n\n### 2.3 특별휴가\n- 출산(본인): 90일(법정 기준에 따름) — 별도 관리\n- 배우자 출산: 10일\n- 자녀 돌봄: 연 3일 (미성년 자녀 보유자만)\n\n## 3. 연봉 상승률 (기본 가이드)\n- 사원: 3~5%\n- 대리: 4~6%\n- 과장: 5~7%\n- 차장: 4~6%\n- 부장: 3~5%\n\n### 3.1 성과 가중치 (Performance Multiplier)\n- A: +2%p\n- B: +0%p\n- C: -1%p (단, 하한 0% 보장)\n\n> 예) 대리(B)의 기본 4~6% → B는 +0%p이므로 4~6% 유지\n> 예) 과장(A)의 기본 5~7% → A는 +2%p → 7~9%'), Document(metadata={'source': 'company_policy.md'}, page_content='> 예) 대리(B)의 기본 4~6% → B는 +0%p이므로 4~6% 유지\n> 예) 과장(A)의 기본 5~7% → A는 +2%p → 7~9%\n\n## 4. 승진(프로모션) 참고\n- 사원→대리: 최소 근속 2년 & 최근 2개년 평균 평가 B 이상\n- 대리→과장: 최소 근속 4년 & 최근 2개년 평균 

In [199]:
from re import search
embedding = OpenAIEmbeddings(model='text-embedding-3-small')
vectordb = Chroma.from_documents(splits, embedding)
retriever = vectordb.as_retriever(search_kwargs = {'k':3})
# 상위 3개로 검색 (k = 3)

## 판다스 에이전트 만들기


In [200]:
from langchain_experimental.agents.agent_toolkits.pandas.base import create_pandas_dataframe_agent
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model = 'gpt-4o-mini')

emp_agent = create_pandas_dataframe_agent(
    llm,
    df,
    verbose = True,
    allow_dangerous_code = True,
    prefix = PREFIX
)

In [201]:
emp_agent.invoke("A등급 사원의 평균 연봉을 계산해줘 (단위: 백만원)")



> Entering new AgentExecutor chain...
Thought: I need to filter the DataFrame for employees with performance grade 'A' and then calculate the average base salary for those employees. 
Action: python_repl_ast
Action Input: df[df['performance'] == 'A']['base_salary_mkrw'].mean()61.333333333333336I now know the final answer
Final Answer: A등급 사원의 평균 연봉은 61.33 백만원입니다.

> Finished chain.


{'input': 'A등급 사원의 평균 연봉을 계산해줘 (단위: 백만원)',
 'output': 'A등급 사원의 평균 연봉은 61.33 백만원입니다.'}

##RAG + 직원 데이터를 통합하는 체인


In [202]:
# RunnableParallel 병렬처리 -> batch()랑 유사

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [203]:
prompt = ChatPromptTemplate.from_template("""
너는 HR 전문가야.
[직원 요약]
{emp_summary}

[정책 근거]
{policy_snippets}

[사용자 질문]
{question}

출력 형식:
- 정책에 대한 요약:
- 계산/규정에 대한 적용:
- 결론(한줄):

""")

## 본격적인 체인 구성

In [204]:
# 1. SQL에 대한 체인 준비
def summarize_employee(name):
  q = f"{name}직원의 직급, 근속연수(years_of_service), 성과등급(performance), 남은 연차(remaining_leave), 기본연봉(base_salary_mkrw)를 한줄로 요약해줘"
  res = emp_agent.invoke(q)
  # Extract the natural language summary from the agent's output
  return res if isinstance(res, str) else res.get("output", str(res))

In [205]:
# 2. 정책에 대한 체인 준비
def search_policy(query):
  return retriever.get_relevant_documents(query)


In [206]:
# 3. retriever로 3가지 추출된 자료에 대해서 1개의 텍스트 파일화
def format_docs(docs):
  return("\n----\n".join([d.page_content[:300] for d in docs]))

## Langchain 구축

In [207]:
# 위에 구축해놓은것 다 한번에 돌아가도록 연결
# RunnableParallel - 두가지 작업을 병렬 실행을 해서 순차적인 실행이 아닌 동시에 서로 다른 두 에이전트가 실행되도록
# RunnablePassthrough.assign policy_docs.변수 -> policy_snippets 변수로 새 필드 적용(할당)
# lambda함수는 summarize_employee, search_policy 위에서 정의한 함수가 실행되도록 구현!

chain = (
    RunnableParallel(
        # SQL 쿼리에서 작성된 직원 정보를 emp_summary로 할당
        emp_summary = lambda x : summarize_employee(x["name"]), # name에 들어오는 고객의 값을 emp_summary에 넣어라
        # search_policy에서 질문에 유사한 답변을 policy_docs에 할당
        policy_docs = lambda x: search_policy(x["question"]),
        question = lambda x: x["question"]
    )
    | RunnablePassthrough.assign(
        policy_snippets = lambda x: format_docs(x["policy_docs"])
    )

    | prompt
    | llm
    | StrOutputParser()

)

In [208]:
query = {
    "name" : "김지훈",
    "question" : "부장직급의 2026년도 연봉상승률과 연차규정을 알려줘"
}
response = chain.invoke(query)
print(response)

# 병렬 처리로 진행해서 가끔 작동이 잘 안될 수 있음



> Entering new AgentExecutor chain...
Thought: 김지훈 직원의 정보를 찾기 위해 df에서 해당 직원을 검색해야 한다. 
Action: python_repl_ast
Action Input: df[df['name'] == '김지훈'][['rank', 'years_of_service', 'performance', 'remaining_leave', 'base_salary_mkrw']]  rank  years_of_service performance  remaining_leave  base_salary_mkrw
8   부장                16           B                3             110.0I now know the final answer
Final Answer: 김지훈 직원의 직급은 부장, 근속연수는 16년, 성과등급은 B, 남은 연차는 3일, 기본연봉은 110.0 백만원입니다.

> Finished chain.
- 정책에 대한 요약: 부장 직급의 기본 연차는 22일이며, 10년 이상의 근속에 대해 +4일의 가산이 적용됩니다. 미사용 연차는 최대 5일까지 이월할 수 있습니다. 연봉 상승률에 대한 구체적인 수치는 내규에서 제시되지 않았으나, 일반적으로 성과등급에 따라 결정됩니다.

- 계산/규정에 대한 적용: 김지훈 부장은 16년 근속으로, 기본 연차 22일에 추가로 +4일이 적용되어 총 26일의 연차를 보유할 수 있습니다. 그러나 최대 이월 가능 일수는 5일이므로 사용하지 않은 연차가 5일 이내일 경우 이월 가능합니다. 연봉 상승률은 성과등급 B에 따라 회사의 연봉 상승 정책에 규정된 비율로 인상될 것으로 예상됩니다.

- 결론(한줄): 김지훈 부장은 2026년도 총 26일의 연차와 성과에 따라 연봉 상승이 기대됩니다.


## Google RSS 기반 웹검색 에이전트 작성

In [209]:
! pip install -q feedparser

In [210]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

In [211]:
# 내부 문서 생성
documents = [
    Document(page_content = "우리회사는 2020년에 설립된 AI 스타트업입니다."),
    Document(page_content = "주력 제품은 고객 상담 챗봇 솔루션입니다."),
    Document(page_content = "현재 직원수는 50명이고, 서울에 본사가 있습니다."),
    Document(page_content = "2024년도 매출은 전년대비 150% 성장하고 있습니다."),
    Document(page_content = "주요 고객사는 kakao, naver, cupang 등 입니다.")
]

In [212]:
# 문서 -> 임베딩 / chroma db / retriever가 필요하다!
emb = OpenAIEmbeddings()
vectorstore = Chroma.from_documents(documents = documents, embedding = emb)
retriever = vectorstore.as_retriever(search_kwargs = {'k':2})


In [213]:
retriever.get_relevant_documents("매출, 설립년월")

[Document(metadata={}, page_content='2024년도 매출은 전년대비 150% 성장하고 있습니다.'),
 Document(metadata={}, page_content='2024년도 매출은 전년대비 150% 성장하고 있습니다.')]

In [214]:
import feedparser
from urllib.parse import quote

rss = f"https://news.google.com/rss/search?q={quote("삼성전자")}&hl=ko&gl=KR&ceid=KR:ko"
feed = feedparser.parse(rss)

print(feed)

{'bozo': False, 'entries': [{'title': '삼성전자, 美 OLED 특허소송서 1억9천만달러 배상 평결(종합) - 연합뉴스', 'title_detail': {'type': 'text/plain', 'language': None, 'base': 'https://news.google.com/rss/search?q=%EC%82%BC%EC%84%B1%EC%A0%84%EC%9E%90&hl=ko&gl=KR&ceid=KR:ko', 'value': '삼성전자, 美 OLED 특허소송서 1억9천만달러 배상 평결(종합) - 연합뉴스'}, 'links': [{'rel': 'alternate', 'type': 'text/html', 'href': 'https://news.google.com/rss/articles/CBMiW0FVX3lxTE5McnBBTFFDSElUbWR6QWNoOUIxT2dmVkU3Z0VqSkxiTkJVR1RnbndOMkxxbTVLSXZ3SHluNUlnSVk3eHQ0WU5jWkNWeWdoQ2VqQjl1SmN3bWVjMFXSAWBBVV95cUxNVEJRWHhoSm8zaFdSXzkyUm9ocGZQM05RazhPTjZCTzFMM3diMkc5M2U3TWVIRk1RUFExb0sxQ0JlTUdMc1lOc2RLeUZnZWFwa2RtQTFWUm9FMmx4OVAyWDg?oc=5'}], 'link': 'https://news.google.com/rss/articles/CBMiW0FVX3lxTE5McnBBTFFDSElUbWR6QWNoOUIxT2dmVkU3Z0VqSkxiTkJVR1RnbndOMkxxbTVLSXZ3SHluNUlnSVk3eHQ0WU5jWkNWeWdoQ2VqQjl1SmN3bWVjMFXSAWBBVV95cUxNVEJRWHhoSm8zaFdSXzkyUm9ocGZQM05RazhPTjZCTzFMM3diMkc5M2U3TWVIRk1RUFExb0sxQ0JlTUdMc1lOc2RLeUZnZWFwa2RtQTFWUm9FMmx4OVAyWDg?oc=5', 'id': 'CBMiW0

In [215]:
import feedparser
from urllib.parse import quote

def fetch_google_news(query, max_results = 3):
    rss = f"https://news.google.com/rss/search?q={quote(query)}&hl=ko&gl=KR&ceid=KR:ko"
    feed = feedparser.parse(rss)
    lines = []
    # title(제목) / link(원문) / summary(요약/미리보기) / publish(계시 시각) / source(원본 매체정보)
    for e in feed.entries[:max_results]:
      # 뉴스에서 title, link, date, source 정보를 가져오고 -> 리스트 -> join 텍스트화
      title = e.get("title", "").strip() # title 있으면 가지고 오고, 없으면 공뱁
      link = e.get("link", "")
      date = e.get("published", "날짜 미상")
      # e 객체에서 source가 있으면 꺼내고, 없으면 None / 있으면  title 속성 가져오고, 없으면 출처 미상
      source = getattr(getattr(e, "source", None), "title", "출처 미상")

      lines.append(f"{title}\n 출처 ; {source} | 날짜 {date} \n 링크 {link}")

    return "\n\n".join(lines)

In [216]:
fetch_google_news("삼성전자")

'삼성전자, 美 OLED 특허소송서 1억9천만달러 배상 평결(종합) - 연합뉴스\n 출처 ; 연합뉴스 | 날짜 Mon, 03 Nov 2025 19:46:11 GMT \n 링크 https://news.google.com/rss/articles/CBMiW0FVX3lxTE5McnBBTFFDSElUbWR6QWNoOUIxT2dmVkU3Z0VqSkxiTkJVR1RnbndOMkxxbTVLSXZ3SHluNUlnSVk3eHQ0WU5jWkNWeWdoQ2VqQjl1SmN3bWVjMFXSAWBBVV95cUxNVEJRWHhoSm8zaFdSXzkyUm9ocGZQM05RazhPTjZCTzFMM3diMkc5M2U3TWVIRk1RUFExb0sxQ0JlTUdMc1lOc2RLeUZnZWFwa2RtQTFWUm9FMmx4OVAyWDg?oc=5\n\n삼성전자, HBM4 핵심 ‘1c D램’ 개발팀에 인센티브 - 조선일보\n 출처 ; 조선일보 | 날짜 Mon, 03 Nov 2025 07:24:01 GMT \n 링크 https://news.google.com/rss/articles/CBMigwFBVV95cUxNS3d0aGpuUjRFelFpSVQwVVZJa2pYWF8zZWRzN2ZsWXFIcjNDZFZWaWVJSWpmbldwYXpkdEYyRGxJZnNUTDNsSHQwMDhtckZ5UTBLcUJwRGp6NVBJV2lTek05SktQVEgxczNVemRhbHVudU12ZGJXYkpycUl4MHpGNXdZTdIBlwFBVV95cUxPNlFjelNhRVNVY3U2VlhQN1lwVktkSU5EQjAtTzJaeWt5MGtDVjd3cnlNUVFXMUhBbWZ2RGtwaEVIX29Bam5BUU9ZWmthb0pwVFhpaEUxOXZNRjZKcGFlTUNGTnhUMkZvNVY2dEU2ajJaMzFMWVNXZjBlYkZrZkt5aEMwTFVxeW96emhPa3BvaEZFOHlRVFd3?oc=5\n\n“9만전자 때 살 걸···” 삼성전자·SK하이닉스 나란히 신고가, 목표주가도 지붕 뚫었다 - v.daum.net\n 출처 ; v

## LCEL 파이프라인 구현

In [217]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough, RunnableParallel

# Formatter 포메터

# 내부문서 요약기 RAG 검색결과를 받아서 page_content 를 한줄의 문자열로 표현하는 구문
format_docs = RunnableLambda(lambda docs : "\n".join(d.page_content for d in docs))

# 뉴스 수집기 -> 헤드라인 3개에 대한 정보를 가져오는 구문
news_tool = RunnableLambda(lambda q : fetch_google_news(q, max_results=3))

# 병렬 실행
gather = RunnableParallel(
    internal = retriever | format_docs, # 내부 문서
    news = news_tool, # 뉴스
    q = RunnablePassthrough() # 원래 질문 그대로 전달
)

# 병렬 실행 예시 삼성전자
# internal : 사내 전략 문서 내용 요약
# news : 오늘자 헤드라인 뉴스 3개
# q : 질문 그대로


In [218]:
# chain 구성하기
prompt= ChatPromptTemplate.from_template("""
아래 정보를 바탕으로 질문에 답하세요.

[내부자료]
{internal}

[뉴스]
{news}

질문 : {q}

규칙 :
- 출처를 명확히 구분해서 표기([내부자료]/[뉴스] 중에서)
- 날짜, 숫자는 정확하게 인용
- 정보가 부족하거나 없으면 "추가정보가 필요합니다."라고 한문장으로 명시해라
- 한국어로 5문장 이내로 작성해라
""")

In [219]:
llm = ChatOpenAI(model = 'gpt-4o-mini', temperature = 0)
chain = gather | prompt | llm

In [220]:
questions = [
   '우리 회사 현재 직원수는?',
   '삼성전자 최신 뉴스를 알려줘',
   '애플 실적 발표 소식',
   '우리 회사와 유사한 2025년 10월 기준 ai 회사 동향'
]

for q in questions :
  print(q)
  print(chain.invoke(q).content)

우리 회사 현재 직원수는?
우리 회사의 현재 직원수는 50명입니다. 이 정보는 내부자료에서 확인할 수 있습니다. 본사는 서울에 위치하고 있습니다. 추가정보가 필요합니다.
삼성전자 최신 뉴스를 알려줘
삼성전자는 2025년 3분기 매출이 86조원에 달하며 분기 사상 최대치를 기록했습니다. ([뉴스] 매일경제, 2025년 10월 30일) 또한, 2025년형 비스포크 AI 세탁기와 건조기를 출시했습니다. ([뉴스] samsung.com, 2025년 7월 17일) 이러한 성과는 삼성전자가 AI 및 스마트 가전 분야에서의 경쟁력을 강화하고 있음을 보여줍니다. 추가정보가 필요합니다.
애플 실적 발표 소식
애플의 실적 발표 소식에 따르면, 애플은 예상보다 좋은 실적을 기록하며 주가가 시간외 거래에서 2% 상승했습니다([뉴스], 2025년 8월 1일). 뉴욕증시에서는 엇갈린 기업 실적에 혼조세를 보였으나, 애플의 투자 확대 소식에 기술주가 반등하는 모습을 보였습니다([뉴스], 2025년 8월 6일). 추가정보가 필요합니다.
우리 회사와 유사한 2025년 10월 기준 ai 회사 동향
2025년 10월 기준으로 AI 분야에서는 여러 가지 동향이 나타나고 있습니다. 예를 들어, OpenAI의 GPT-5는 정치적 편견을 30% 줄이는 성과를 보였으며, 이는 AI 기술의 발전을 보여줍니다([뉴스] AI넷, 2025년 10월 19일). 또한, Thinking Machines Lab의 공동 창립자가 Meta에 합류하여 비디오 게임 생성을 위한 xAI의 세계 모델 Agent Builder를 8,000개 이상의 도구에 연결하는 등의 혁신이 이루어지고 있습니다([뉴스] AI넷, 2025년 10월 19일). 이러한 동향은 AI 스타트업들이 기술적 진보와 협업을 통해 시장에서 경쟁력을 높이고 있음을 시사합니다. 추가정보가 필요합니다.
